# Heavy Equipment Selling Price Prediction — competition notebook

Original Kaggle notebook **`24f3002375-notebook-2026t2`, Version 23** — the submission that scored
**RMSLE 0.18741** on the leaderboard (rank 85 / 2,593 teams).

This is kept as the record of the work done during the competition. The same logic, refactored into a
tested Python package, lives in `src/heavy_equipment/` and is run with `python scripts/train.py`.

> Exploratory print-outs (`combined.columns`, `.head()`) are kept as in the original; the outputs of the
> key cells (shapes, fold scores, blend weights) are preserved below them.

Data paths point at the Kaggle input folder. To run locally, change `DATA_DIR` to `../data/raw`.

In [ ]:
import numpy as np
import pandas as pd
import warnings
import math
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from scipy.optimize import minimize

In [ ]:
DATA_DIR = '/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge'
train = pd.read_csv(f'{DATA_DIR}/train.csv', low_memory=False)
test = pd.read_csv(f'{DATA_DIR}/test.csv', low_memory=False)

print(f'Train SHape: {train.shape} \n  Test Shape: {test.shape}')

Train SHape: (138701, 50)
  Test Shape: (15000, 49)


In [ ]:
train.columns

In [ ]:
test.columns

In [ ]:
valid_prices = np.sort(train['TargetValue'].unique())
target     = train['TargetValue'].values
log_target = np.log1p(target)       # log1p(x) = log(1 + x)

train_ids = train['TransactionID'].values
test_ids  = test['TransactionID'].values

df_train = train.drop(['TargetValue'], axis=1).copy()
df_test  = test.copy()


df_train['_is_test'] = 0
df_test['_is_test']  = 1


combined  = pd.concat([df_train, df_test], ignore_index=True)
train_len = len(df_train)

# y = log-transformed price for training rows only
y = pd.Series(log_target, name='log_target')

In [ ]:
train['TargetValue'].head(10)

In [ ]:
combined.head(10)

In [ ]:
combined.columns

In [ ]:
log_target

In [ ]:
for col in ['ProductConfigID', 'AssetID']:
    combined[col] = combined[col].astype(str)

In [ ]:
missing_ratio = combined.isnull().mean()
missing_ratio

In [ ]:
drop_cols = []
for col in missing_ratio.index:
    if missing_ratio[col] > 0.99:
        if col != 'TransactionID' and col != '_is_test':
            drop_cols.append(col)

print(f'Dropped: {drop_cols}')

combined.drop(columns=drop_cols, inplace=True)

Dropped: ['col18', 'col19']


In [ ]:
combined.columns

Date Feature

In [ ]:
combined['TransactionDate'] = pd.to_datetime(combined['TransactionDate'], errors='coerce')
combined['TransactionDate'].head(2)

In [ ]:
combined['SaleYear']        = combined['TransactionDate'].dt.year
combined['SaleMonth']       = combined['TransactionDate'].dt.month
combined['SaleDay']         = combined['TransactionDate'].dt.day

In [ ]:
combined['SaleMonth']

In [ ]:
combined['SaleDayOfWeek']   = combined['TransactionDate'].dt.dayofweek
combined['SaleDayOfWeek'].head(2)

In [ ]:
combined['SaleQuarter']     = combined['TransactionDate'].dt.quarter
combined['SaleDayOfYear']   = combined['TransactionDate'].dt.dayofyear

In [ ]:
min_date = combined['TransactionDate'].min()
min_date

In [ ]:
combined['DaysSinceStart']  = (combined['TransactionDate'] - min_date).dt.days
combined['SaleIsYearEnd']   = (combined['SaleMonth'] == 12).astype(int)
combined['SaleIsYearStart'] = (combined['SaleMonth'] == 1).astype(int)
combined['SaleDecade']      = (combined['SaleYear'] // 10).astype(int)
combined['SaleYear_sq']     = combined['SaleYear'] ** 2
combined.drop('TransactionDate', axis=1, inplace=True)

In [ ]:
combined['DaysSinceStart'].head(2)

In [ ]:
combined.columns

MAnufacture Year and Equipment Age

In [ ]:
combined['ManufactureYear'] = combined['ManufactureYear'].replace({1001: np.nan, 1000: np.nan})

combined.loc[combined['ManufactureYear'] < 1900, 'ManufactureYear'] = np.nan
combined.loc[combined['ManufactureYear'] > 2030, 'ManufactureYear'] = np.nan

combined['ManufactureYear_missing'] = combined['ManufactureYear'].isnull().astype(int)

combined['ManufactureYear_missing']

In [ ]:
yr_med = combined.iloc[:train_len]['ManufactureYear'].median()
print(yr_med)

1999.0


In [ ]:
combined['ManufactureYear'] = combined['ManufactureYear'].fillna(yr_med)

combined['EquipmentAge']  = (combined['SaleYear'] - combined['ManufactureYear']).clip(0, 65)
combined['Log_Age']       = np.log1p(combined['EquipmentAge'])
combined['Sqrt_Age']      = np.sqrt(combined['EquipmentAge'])
combined['Age_Squared']   = combined['EquipmentAge'] ** 2
combined['Age_Decade']    = (combined['EquipmentAge'] // 10).astype(int)

In [ ]:
combined.columns

In [ ]:
combined['Age_Bucket']    = pd.cut(combined['EquipmentAge'],
                                   bins=[-1, 2, 5, 10, 20, 65],
                                   labels=[0, 1, 2, 3, 4]).astype(int)
combined['Age_Bucket']

In [ ]:
combined['IsNew']         = (combined['EquipmentAge'] <= 2).astype(int)
combined['IsOld']         = (combined['EquipmentAge'] >= 20).astype(int)

combined['IsNew']

Operational Hour Meter

In [ ]:
combined['OperationalHoursMeter'] = combined['OperationalHoursMeter'].replace(0, np.nan)
combined['Hours_missing']      = combined['OperationalHoursMeter'].isnull().astype(int)
combined[['OperationalHoursMeter','Hours_missing']].head(10)

In [ ]:
hrs_med = combined.iloc[:train_len]['OperationalHoursMeter'].median()
hrs_med

In [ ]:
combined['OperationalHoursMeter'] = combined['OperationalHoursMeter'].fillna(hrs_med)
combined[['OperationalHoursMeter','Hours_missing']].head(10)

In [ ]:
combined['Log_Hours']          = np.log1p(combined['OperationalHoursMeter'])
combined['Sqrt_Hours']         = np.sqrt(combined['OperationalHoursMeter'])

combined['Hours_per_year']     = combined['OperationalHoursMeter'] / (combined['EquipmentAge'] + 1)
combined['Log_Hours_per_year'] = np.log1p(combined['Hours_per_year'])

combined['Hours_x_Age']        = combined['Log_Hours'] * combined['Log_Age']
combined['Hours_x_SaleYear']   = combined['Log_Hours'] * combined['SaleYear']

In [ ]:
combined.columns

In [ ]:
import re

print("Extracting hidden specifications and calculating relative ratios...")


def extract_spec_number(text):
    if pd.isna(text):
        return np.nan

    match = re.search(r'(\d+\.?\d*)', str(text))
    return float(match.group(1)) if match else np.nan

if 'Spec_FullDescriptor' in combined.columns:
    combined['Spec_Extracted_Num'] = combined['Spec_FullDescriptor'].apply(extract_spec_number)


    spec_med = combined.iloc[:train_len]['Spec_Extracted_Num'].median()
    combined['Spec_Extracted_Num'] = combined['Spec_Extracted_Num'].fillna(spec_med)


    combined['Spec_Num_x_Age'] = combined['Spec_Extracted_Num'] * combined['EquipmentAge']


if 'ProductConfigID' in combined.columns:
    cfg_med_hours = combined.groupby('ProductConfigID')['OperationalHoursMeter'].transform('median')

    combined['Hours_Ratio_vs_Config'] = combined['OperationalHoursMeter'] / (cfg_med_hours + 1)

if 'Spec_BaseClass' in combined.columns:
    base_med_hours = combined.groupby('Spec_BaseClass')['OperationalHoursMeter'].transform('median')
    combined['Hours_Ratio_vs_BaseClass'] = combined['OperationalHoursMeter'] / (base_med_hours + 1)

Utilization Tier Ordinal

In [ ]:
combined['UtilizationTier']

In [ ]:
if 'UtilizationTier' in combined.columns:
    util_map = {'Low': 0, 'Medium': 1, 'High': 2}
    combined['UtilizationTier'] = combined['UtilizationTier'].fillna('Unknown')
    print(combined['UtilizationTier'])
    combined['UtilizationTier_ord'] = combined['UtilizationTier'].map(util_map).fillna(-1)
combined['UtilizationTier_ord']

In [ ]:
combined[['UtilizationTier','UtilizationTier_ord']]

Interaction Compound Columns

All these lines create interaction (cross) features by joining two categorical variables into one string. These new features help our ML model to capture relationships between variables that would not be visible if each variable were used independently.

In [ ]:
combined['Cfg_AgeBucket']   = combined['ProductConfigID'].astype(str) + '_' + combined['Age_Bucket'].astype(str)
combined[['ProductConfigID','Age_Bucket','Cfg_AgeBucket']]

In [ ]:
combined['Cfg_SaleDecade']  = combined['ProductConfigID'].astype(str) + '_' + combined['SaleDecade'].astype(str)
combined['Cfg_SaleYear']    = combined['ProductConfigID'].astype(str) + '_' + combined['SaleYear'].astype(str)
combined['Base_AgeBucket']  = combined['Spec_BaseClass'].astype(str)  + '_' + combined['Age_Bucket'].astype(str)
combined['Base_SaleDecade'] = combined['Spec_BaseClass'].astype(str)  + '_' + combined['SaleDecade'].astype(str)
combined['Func_AgeBucket']  = combined['FunctionalClassification'].astype(str) + '_' + combined['Age_Bucket'].astype(str)
combined['InvCat_SaleYear'] = combined['InventoryGroupCategory'].astype(str) + '_' + combined['SaleYear'].astype(str)
combined['Region_InvCat']   = combined['RegionCode'].astype(str) + '_' + combined['InventoryGroupCategory'].astype(str)
combined['Vendor_InvCat']   = combined['VendorPartnerID'].astype(str) + '_' + combined['InventoryGroupCategory'].astype(str)
combined['Scale_InvCat']    = combined['AssetScaleFactor'].astype(str) + '_' + combined['InventoryGroupCategory'].astype(str)
combined['Cfg_Scale']       = combined['ProductConfigID'].astype(str) + '_' + combined['AssetScaleFactor'].astype(str)

In [ ]:
combined.columns

Non-target Aggregate Features

In [ ]:
group_columns = [
    'ProductConfigID',
    'Spec_BaseClass',
    'FunctionalClassification',
    'InventoryGroupCategory',
    'RegionCode',
    'Cfg_AgeBucket'
]

for group_col in group_columns:

    # Skip if the column does not exist
    if group_col not in combined.columns:
        continue

    # Statistics for OperationalHoursMeter

    hours_stats = combined.groupby(group_col)['OperationalHoursMeter'].agg(
        ['mean', 'median', 'std']
    )

    hours_stats = hours_stats.rename(columns={
        'mean': group_col + '_Hours_mean',
        'median': group_col + '_Hours_median',
        'std': group_col + '_Hours_std'
    })

    combined = combined.merge(hours_stats, on=group_col, how='left')


    # Statistics for EquipmentAge

    age_stats = combined.groupby(group_col)['EquipmentAge'].agg(
        ['mean', 'std', 'min', 'max']
    )

    age_stats = age_stats.rename(columns={
        'mean': group_col + '_Age_mean',
        'std': group_col + '_Age_std',
        'min': group_col + '_Age_min',
        'max': group_col + '_Age_max'
    })

    combined = combined.merge(age_stats, on=group_col, how='left')


    # Count how many rows belong to each group

    group_count = combined.groupby(group_col).size()

    group_count = group_count.rename(group_col + '_Count')

    combined = combined.merge(group_count, on=group_col, how='left')

In [ ]:
combined.columns

Relative Features

In [ ]:
combined['Hours_pct_in_cfg']  = combined.groupby('ProductConfigID')['OperationalHoursMeter'].rank(pct=True)
combined['Age_pct_in_cfg']    = combined.groupby('ProductConfigID')['EquipmentAge'].rank(pct=True)
combined['Day_pct_in_cfg']    = combined.groupby('ProductConfigID')['DaysSinceStart'].rank(pct=True)

cfg_med_hours = combined.groupby('ProductConfigID')['Log_Hours'].transform('median')
combined['Hours_dev_cfg']     = combined['Log_Hours'] - cfg_med_hours

cfg_med_age = combined.groupby('ProductConfigID')['EquipmentAge'].transform('median')
combined['Age_dev_cfg']       = combined['EquipmentAge'] - cfg_med_age

cfg_med_day = combined.groupby('ProductConfigID')['DaysSinceStart'].transform('median')
combined['Day_dev_cfg']       = combined['DaysSinceStart'] - cfg_med_day

Filling Missing Values

In [ ]:
EXCLUDE = {'_is_test', 'TransactionID'}
cat_cols = [c for c in combined.select_dtypes(include=['object']).columns if c not in EXCLUDE]
num_cols = [c for c in combined.select_dtypes(include=[np.number]).columns if c not in EXCLUDE]

for col in cat_cols:
    combined[col] = combined[col].fillna('Unknown')

for col in num_cols:
    if combined[col].isnull().any():
        miss_flag = f'{col}_miss'
        if miss_flag not in combined.columns:
            combined[miss_flag] = combined[col].isnull().astype(int)
        combined[col] = combined[col].fillna(combined.iloc[:train_len][col].median())

Target Encoding

In [ ]:
print(y)

In [ ]:
kf_te = KFold(n_splits=5, shuffle=True, random_state=2024)
global_mean = y.mean()
print(global_mean)
SMOOTH = 10

10.424110798973885


In [ ]:
kf_te

In [ ]:
col

In [ ]:
combined['Day_dev_cfg'].head(20)

In [ ]:
combined['Day_dev_cfg'].value_counts()/len(combined)

In [ ]:
def te_encode(combined, col, y, train_len, kf_te, global_mean, smooth):
    freq = combined[col].value_counts() / len(combined)

    combined[f'{col}_freq'] = combined[col].map(freq)
    combined[f'{col}_te'] = np.nan

    for tr_idx, val_idx in kf_te.split(range(train_len)):
        ct = combined.iloc[tr_idx][col].values
        cv = combined.iloc[val_idx][col].values
        yt = y.iloc[tr_idx].values
        s  = pd.DataFrame({'c': ct, 'y': yt}).groupby('c')['y'].agg(['mean', 'count'])
        sm = (s['mean'] * s['count'] + global_mean * smooth) / (s['count'] + smooth)
        combined.loc[val_idx, f'{col}_te'] = pd.Series(cv).map(sm).values
    sf = pd.DataFrame({'c': combined.iloc[:train_len][col].values, 'y': y.values})
    sf = sf.groupby('c')['y'].agg(['mean', 'count'])
    smf = (sf['mean'] * sf['count'] + global_mean * smooth) / (sf['count'] + smooth)
    combined.loc[train_len:, f'{col}_te'] = combined.loc[train_len:, col].map(smf).values
    combined[f'{col}_te'].fillna(global_mean, inplace=True)
    return combined

for col in cat_cols:
    combined = te_encode(combined, col, y, train_len, kf_te, global_mean, SMOOTH)

Label Encode Categoricals

In [ ]:
le = LabelEncoder()
for col in cat_cols:
    combined[col] = combined[col].astype(str)
    combined[col] = le.fit_transform(combined[col])

Numeric Interaction Features

In [ ]:
for cat_col in ['InventoryGroupCategory', 'FunctionalClassification', 'Spec_BaseClass', 'RegionCode']:
    if cat_col not in combined.columns: continue
    for num_col in ['EquipmentAge', 'Log_Hours', 'SaleYear']:
        combined[f'{cat_col[:6]}_x_{num_col[:6]}'] = combined[cat_col] * combined[num_col]

Train Test Split

In [ ]:
X_train = combined[combined['_is_test'] == 0].drop(['_is_test', 'TransactionID'], axis=1)
X_test  = combined[combined['_is_test'] == 1].drop(['_is_test', 'TransactionID'], axis=1)
print(f'Features: {X_train.shape[1]}')

def rmsle(y_true, y_pred):
    return math.sqrt(mean_squared_error(np.array(y_true), np.array(y_pred)))

Features: 270


OOF Training

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb1 = np.zeros(len(X_train)); test_lgb1 = np.zeros(len(X_test))
oof_lgb2 = np.zeros(len(X_train)); test_lgb2 = np.zeros(len(X_test))
oof_xgb  = np.zeros(len(X_train)); test_xgb  = np.zeros(len(X_test))
oof_cat  = np.zeros(len(X_train)); test_cat  = np.zeros(len(X_test))

LightBGM

In [ ]:
# --- LightGBM v1 ---
print('\n=== LightGBM v1 (leaves=255) ===')
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx],       y.iloc[val_idx]
    m = lgb.LGBMRegressor(
        n_estimators=10000, learning_rate=0.008, num_leaves=255, max_depth=-1,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
        min_child_samples=20, reg_alpha=0.05, reg_lambda=0.2,
        random_state=42 + fold, device='cpu', n_jobs=-1, verbose=-1)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb1[val_idx] = m.predict(X_val)
    test_lgb1 += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold+1}: {rmsle(y_val, oof_lgb1[val_idx]):.4f}  iter={m.best_iteration_}')
print(f'LGB1 OOF: {rmsle(y, oof_lgb1):.4f}')


=== LightGBM v1 (leaves=255) ===
  Fold 1: 0.1927  iter=9983
  Fold 2: 0.1945  iter=9998
  Fold 3: 0.1949  iter=9998
  Fold 4: 0.1957  iter=9993
  Fold 5: 0.1941  iter=9994
LGB1 OOF: 0.1944


In [ ]:
# --- LightGBM v2 ---
print('\n=== LightGBM v2 (leaves=511) ===')
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx],       y.iloc[val_idx]
    m = lgb.LGBMRegressor(
        n_estimators=10000, learning_rate=0.006, num_leaves=511, max_depth=-1,
        subsample=0.7, subsample_freq=1, colsample_bytree=0.6,
        min_child_samples=30, reg_alpha=0.1, reg_lambda=0.5,
        random_state=100 + fold, device='cpu', n_jobs=-1, verbose=-1)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb2[val_idx] = m.predict(X_val)
    test_lgb2 += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold+1}: {rmsle(y_val, oof_lgb2[val_idx]):.4f}  iter={m.best_iteration_}')
print(f'LGB2 OOF: {rmsle(y, oof_lgb2):.4f}')


=== LightGBM v2 (leaves=511) ===
  Fold 1: 0.1927  iter=8553
  Fold 2: 0.1949  iter=8205
  Fold 3: 0.1951  iter=7062
  Fold 4: 0.1962  iter=7505
  Fold 5: 0.1943  iter=7336
LGB2 OOF: 0.1947


XGBoost

In [ ]:
print('\n=== XGBoost ===')
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx],       y.iloc[val_idx]
    m = xgb.XGBRegressor(
        n_estimators=10000, learning_rate=0.008, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.05, reg_lambda=1.0,
        min_child_weight=5, random_state=42 + fold, n_jobs=-1,
        early_stopping_rounds=300, eval_metric='rmse', verbosity=0,
        device='cpu', tree_method='hist')
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m.predict(X_val)
    test_xgb += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold+1}: {rmsle(y_val, oof_xgb[val_idx]):.4f}  iter={m.best_iteration}')
print(f'XGB OOF: {rmsle(y, oof_xgb):.4f}')


=== XGBoost ===
  Fold 1: 0.1962  iter=9999
  Fold 2: 0.1979  iter=9998
  Fold 3: 0.1982  iter=9999
  Fold 4: 0.1988  iter=9998
  Fold 5: 0.1974  iter=9999
XGB OOF: 0.1977


CatBoost

In [ ]:
print('\n=== CatBoost ===')
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx],       y.iloc[val_idx]
    m = CatBoostRegressor(
        iterations=8000, learning_rate=0.02, depth=8, l2_leaf_reg=3,
        bootstrap_type='Bernoulli', subsample=0.8, random_seed=42 + fold,
        loss_function='RMSE', early_stopping_rounds=300,
        task_type='CPU', verbose=False)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
    oof_cat[val_idx] = m.predict(X_val)
    test_cat += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold+1}: {rmsle(y_val, oof_cat[val_idx]):.4f}')
print(f'CAT OOF: {rmsle(y, oof_cat):.4f}')


=== CatBoost ===
  Fold 1: 0.1973
  Fold 2: 0.1998
  Fold 3: 0.1997
  Fold 4: 0.2005
  Fold 5: 0.1988
CAT OOF: 0.1992


Ridge

In [ ]:
from sklearn.linear_model import BayesianRidge

print('\n=== Bayesian Ridge Stacker ===')
L2_train = np.column_stack([oof_lgb1, oof_lgb2, oof_xgb, oof_cat])
L2_test  = np.column_stack([test_lgb1, test_lgb2, test_xgb, test_cat])

oof_stack  = np.zeros(len(X_train))
test_stack = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):

    stk = BayesianRidge()
    stk.fit(L2_train[tr_idx], y.iloc[tr_idx])

    oof_stack[val_idx] = stk.predict(L2_train[val_idx])
    test_stack += stk.predict(L2_test) / N_FOLDS

    print(f'  Fold {fold+1}: {rmsle(y.iloc[val_idx], oof_stack[val_idx]):.4f}')

print(f'Stack OOF: {rmsle(y, oof_stack):.4f}')


=== Bayesian Ridge Stacker ===
  Fold 1: 0.1919
  Fold 2: 0.1942
  Fold 3: 0.1943
  Fold 4: 0.1953
  Fold 5: 0.1935
Stack OOF: 0.1938


Optimize Ensemble Weights

In [ ]:
def neg_rmsle(w):
    w = np.abs(w)
    s = w.sum()
    if s < 1e-10: return 1e10
    w /= s
    blend = w[0]*oof_lgb1 + w[1]*oof_lgb2 + w[2]*oof_xgb + w[3]*oof_cat + w[4]*oof_stack
    return rmsle(y, blend)

res = minimize(neg_rmsle, [1, 1, 0.5, 0.5, 2], method='Nelder-Mead',
               options={'maxiter': 30000, 'xatol': 1e-7, 'fatol': 1e-7})
opt = np.abs(res.x)
opt /= opt.sum()
w1, w2, w3, w4, w5 = opt
print(f'\nWeights: LGB1={w1:.3f} LGB2={w2:.3f} XGB={w3:.3f} CAT={w4:.3f} Stack={w5:.3f}')
print(f'Blended OOF RMSLE: {neg_rmsle(opt):.4f}')


Weights: LGB1=0.068 LGB2=0.073 XGB=0.016 CAT=0.020 Stack=0.822
Blended OOF RMSLE: 0.1938


Predictions

In [ ]:
test_log   = w1*test_lgb1 + w2*test_lgb2 + w3*test_xgb + w4*test_cat + w5*test_stack
test_preds = np.expm1(test_log)


test_preds_rounded = np.round(test_preds / 500) * 500
test_preds_rounded = np.clip(test_preds_rounded, valid_prices.min(), valid_prices.max())


oof_blend     = w1*oof_lgb1 + w2*oof_lgb2 + w3*oof_xgb + w4*oof_cat + w5*oof_stack
oof_raw       = np.expm1(oof_blend)
oof_snap      = np.clip(np.round(oof_raw / 500) * 500, valid_prices.min(), valid_prices.max())

rmsle_raw     = rmsle(y, oof_blend)
rmsle_snap    = rmsle(y, np.log1p(oof_snap))

print(f'\nOOF RMSLE (raw):    {rmsle_raw:.4f}')
print(f'OOF RMSLE (snapped): {rmsle_snap:.4f}')

final_preds = test_preds_rounded if rmsle_snap < rmsle_raw else test_preds.clip(0)

sub = pd.DataFrame({'TransactionID': test_ids, 'TargetValue': final_preds})
sub.to_csv('submission.csv', index=False)

print(sub['TargetValue'].describe())


OOF RMSLE (raw):    0.1938
OOF RMSLE (snapped): 0.1939
count     15000.000000
mean      41214.743642
std       25504.461891
min        7445.481678
25%       20631.647377
50%       34763.478372
75%       57061.098441
max      141983.021406
Name: TargetValue, dtype: float64
